[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_10/notebook_10_2_lime_explanations.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 10.2: LIME - Local Interpretable Model-agnostic Explanations

**Chapter 10: Interpretability and Explainability in Healthcare AI**

## Introduction: When You Need Quick Local Explanations

LIME (Local Interpretable Model-agnostic Explanations) offers an intuitive approach to explaining any machine learning model:

**Core Idea**: Complex models may be globally incomprehensible, but around any single prediction, they can be approximated by a simple, interpretable model (like linear regression).

### Why LIME?

1. **Model-agnostic**: Works with any black box (neural nets, ensembles, even proprietary APIs)
2. **Fast**: Generates explanations quickly (good for real-time clinical use)
3. **Intuitive**: Linear approximations are easy to understand
4. **Local fidelity**: Accurately represents model behavior near the prediction of interest

### LIME vs SHAP

| Aspect | LIME | SHAP |
|--------|------|------|
| **Theoretical foundation** | Heuristic (local approximation) | Rigorous (game theory) |
| **Speed** | Fast | Slower (except TreeSHAP) |
| **Consistency** | May vary for identical inputs | Guaranteed consistent |
| **Interpretability** | Very intuitive (linear model) | Requires understanding Shapley values |
| **Image data** | Excellent (superpixels) | Limited |

**When to use LIME**:
- Quick prototyping and demos
- Explaining to non-technical stakeholders  
- Image/text data (LIME excels here)
- Black box APIs (no model access needed)

---

## Part 1: Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from lime import lime_tabular
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("✓ Libraries imported")
print("\nThis notebook demonstrates LIME for explaining clinical AI predictions.\n")

In [ ]:
# Create diabetes readmission dataset
def create_diabetes_readmission_dataset(n_samples=3000):
    """
    Predict 30-day hospital readmission for diabetes patients.
    
    Key risk factors:
    - A1C (glycemic control)
    - Number of medications
    - Previous admissions
    - Complications (DKA, hypoglycemia)
    - Discharge disposition
    - Insurance status
    """
    data = []
    
    for i in range(n_samples):
        # Demographics
        age = np.random.normal(65, 12)
        age = np.clip(age, 30, 90)
        
        # Clinical features
        a1c = np.random.normal(8.0, 2.0)  # Glycemic control
        a1c = np.clip(a1c, 5.0, 14.0)
        
        num_medications = int(np.random.poisson(8))
        num_medications = np.clip(num_medications, 1, 20)
        
        time_in_hospital = int(np.random.poisson(4))
        time_in_hospital = np.clip(time_in_hospital, 1, 14)
        
        num_procedures = int(np.random.poisson(2))
        num_lab_procedures = int(np.random.poisson(40))
        
        # Previous history
        num_prior_admissions = int(np.random.poisson(2))
        num_emergency_visits = int(np.random.poisson(1))
        
        # Complications
        had_dka = np.random.random() < 0.15  # Diabetic ketoacidosis
        had_hypoglycemia = np.random.random() < 0.20
        
        # Discharge
        discharge_to_facility = np.random.random() < 0.30
        
        # Insurance
        has_insurance = np.random.random() < 0.85
        
        # Calculate readmission risk
        base_risk = 0.15
        
        # A1C effect (poor control)
        if a1c > 9.0:
            base_risk *= 2.0
        elif a1c > 7.5:
            base_risk *= 1.5
        
        # Polypharmacy
        if num_medications > 12:
            base_risk *= 1.6
        
        # Prior admissions
        if num_prior_admissions > 2:
            base_risk *= 1.8
        
        # Complications
        if had_dka:
            base_risk *= 1.7
        if had_hypoglycemia:
            base_risk *= 1.4
        
        # Discharge to facility (proxy for severity)
        if discharge_to_facility:
            base_risk *= 1.5
        
        # No insurance (access barrier)
        if not has_insurance:
            base_risk *= 1.6
        
        # Cap risk
        base_risk = min(base_risk, 0.70)
        
        readmitted = np.random.random() < base_risk
        
        data.append({
            'age': age,
            'a1c': a1c,
            'num_medications': num_medications,
            'time_in_hospital': time_in_hospital,
            'num_procedures': num_procedures,
            'num_lab_procedures': num_lab_procedures,
            'num_prior_admissions': num_prior_admissions,
            'num_emergency_visits': num_emergency_visits,
            'had_dka': int(had_dka),
            'had_hypoglycemia': int(had_hypoglycemia),
            'discharge_to_facility': int(discharge_to_facility),
            'has_insurance': int(has_insurance),
            'readmitted_30d': int(readmitted)
        })
    
    return pd.DataFrame(data)

# Create dataset
df = create_diabetes_readmission_dataset(n_samples=3000)

print("Diabetes Readmission Dataset Created")
print(f"\nTotal samples: {len(df):,}")
print(f"Readmission rate: {df['readmitted_30d'].mean():.1%}")
print("\nFirst few rows:")
df.head(10)

In [ ]:
# Prepare data
feature_cols = [col for col in df.columns if col != 'readmitted_30d']
X = df[feature_cols].values
y = df['readmitted_30d'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Features: {feature_cols}")

## Part 2: Train Black Box Model

In [ ]:
# Train Gradient Boosting model
print("Training Gradient Boosting model...\n")

model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("✓ Model trained")
print(f"\nTest Performance:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred):.1%}")
print(f"  AUC: {roc_auc_score(y_test, y_prob):.3f}\n")

## Part 3: LIME Explanations

### How LIME Works

1. **Select patient to explain**: Choose a prediction to explain
2. **Generate perturbed samples**: Create synthetic patients by randomly varying features
3. **Get black box predictions**: Run the black box model on synthetic patients
4. **Weight by proximity**: Samples closer to the original patient get higher weight
5. **Fit interpretable model**: Train linear regression on weighted samples
6. **Extract coefficients**: Linear model coefficients = feature importance

In [ ]:
# Initialize LIME explainer
print("Initializing LIME explainer...\n")

explainer = lime_tabular.LimeTabularExplainer(
    training_data=X_train,
    feature_names=feature_cols,
    class_names=['No Readmission', 'Readmitted'],
    mode='classification',
    random_state=42
)

print("✓ LIME explainer initialized\n")

### Explain High-Risk Patient

In [ ]:
# Select high-risk patient
high_risk_patients = np.where(y_prob > 0.6)[0]

if len(high_risk_patients) > 0:
    patient_idx = high_risk_patients[0]
else:
    patient_idx = np.argmax(y_prob)

patient_features = X_test[patient_idx]
patient_prob = y_prob[patient_idx]
patient_true = y_test[patient_idx]

print("="*80)
print("HIGH-RISK PATIENT ANALYSIS")
print("="*80 + "\n")

print(f"Patient ID: {patient_idx}")
print(f"AI Prediction: {'HIGH RISK' if patient_prob > 0.5 else 'LOW RISK'} ({patient_prob:.1%} readmission probability)")
print(f"True outcome: {'Readmitted' if patient_true == 1 else 'Not readmitted'}")
print(f"\nPatient Profile:")

for feat_name, feat_val in zip(feature_cols, patient_features):
    if feat_name in ['had_dka', 'had_hypoglycemia', 'discharge_to_facility', 'has_insurance']:
        display = 'Yes' if feat_val == 1 else 'No'
    elif 'num' in feat_name or 'time' in feat_name:
        display = f"{int(feat_val)}"
    else:
        display = f"{feat_val:.1f}"
    
    print(f"  {feat_name}: {display}")

print("\n" + "="*80)

In [ ]:
# Generate LIME explanation
print("\nGenerating LIME explanation...")
print("(Creating 5000 perturbed samples around this patient)\n")

explanation = explainer.explain_instance(
    data_row=patient_features,
    predict_fn=model.predict_proba,
    num_features=10,
    num_samples=5000
)

print("✓ LIME explanation generated\n")

In [ ]:
# Visualize explanation
fig = explanation.as_pyplot_figure(label=1)
plt.title(f'LIME Explanation: Patient {patient_idx} ({patient_prob:.1%} Readmission Risk)', 
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(f'lime_explanation_patient_{patient_idx}.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 How to Read LIME Explanation:")
print("\n  - Each bar shows one feature's contribution")
print("  - Orange bars = increase readmission risk")
print("  - Blue bars = decrease readmission risk")
print("  - Numbers show feature values for this patient")
print("  - Bar length = magnitude of contribution\n")

In [ ]:
# Extract explanation details
print("\n" + "="*80)
print("CLINICAL INTERPRETATION (LIME Local Linear Approximation)")
print("="*80 + "\n")

print(f"For patients SIMILAR to this one, the key drivers are:\n")

# Get feature contributions
lime_values = explanation.as_list(label=1)

# Separate positive and negative contributors
positive = [(feat, weight) for feat, weight in lime_values if weight > 0]
negative = [(feat, weight) for feat, weight in lime_values if weight < 0]

positive_sorted = sorted(positive, key=lambda x: x[1], reverse=True)
negative_sorted = sorted(negative, key=lambda x: abs(x[1]), reverse=True)

print("Factors INCREASING readmission risk:\n")
for i, (feat_desc, weight) in enumerate(positive_sorted[:5], 1):
    print(f"  {i}. {feat_desc}")
    print(f"     Contribution: +{weight:.3f}")
    
    # Clinical context
    if 'a1c' in feat_desc.lower():
        print("     💡 Poor glycemic control increases readmission risk")
    elif 'medication' in feat_desc.lower():
        print("     💡 Polypharmacy suggests complex medical needs")
    elif 'admission' in feat_desc.lower():
        print("     💡 Prior admissions predict future admissions")
    elif 'dka' in feat_desc.lower():
        print("     💡 Recent DKA indicates unstable diabetes")
    elif 'insurance' in feat_desc.lower():
        print("     💡 Lack of insurance = access barrier to outpatient care")
    print()

if len(negative_sorted) > 0:
    print("Factors DECREASING readmission risk:\n")
    for i, (feat_desc, weight) in enumerate(negative_sorted[:3], 1):
        print(f"  {i}. {feat_desc}")
        print(f"     Contribution: {weight:.3f}")
        print()

print("="*80)

# Get prediction probabilities from LIME's local model
local_pred = explanation.local_pred[1]
print(f"\nLIME's local linear model prediction: {local_pred:.1%}")
print(f"Actual black box model prediction: {patient_prob:.1%}")
print(f"\nFidelity (how well LIME approximates black box): ", end="")
if abs(local_pred - patient_prob) < 0.05:
    print("✅ Excellent (difference < 5%)")
elif abs(local_pred - patient_prob) < 0.10:
    print("✓ Good (difference < 10%)")
else:
    print("⚠️ Fair (difference > 10%, local approximation may be inaccurate)\n")

### Compare Multiple Patients

In [ ]:
# Explain multiple patients for comparison
print("\nComparing LIME explanations across 3 patients...\n")

# Select 3 diverse patients
high_risk_idx = high_risk_patients[0] if len(high_risk_patients) > 0 else np.argmax(y_prob)
low_risk_idx = np.argmin(y_prob)
mid_risk_idx = np.argsort(np.abs(y_prob - 0.5))[0]

selected_patients = [low_risk_idx, mid_risk_idx, high_risk_idx]
risk_labels = ['Low Risk', 'Medium Risk', 'High Risk']

explanations = []

for idx, label in zip(selected_patients, risk_labels):
    exp = explainer.explain_instance(
        X_test[idx],
        model.predict_proba,
        num_features=8,
        num_samples=3000
    )
    explanations.append(exp)
    print(f"✓ {label} patient (probability: {y_prob[idx]:.1%})")

print("\n" + "="*80)
print("COMPARISON: What Drives Different Risk Levels?")
print("="*80 + "\n")

for exp, label, idx in zip(explanations, risk_labels, selected_patients):
    print(f"{label.upper()} ({y_prob[idx]:.1%} readmission risk):")
    lime_vals = exp.as_list(label=1)
    top3_pos = [f for f, w in sorted(lime_vals, key=lambda x: x[1], reverse=True)[:3] if w > 0]
    
    if top3_pos:
        print("  Top contributors:")
        for f in top3_pos:
            print(f"    • {f}")
    else:
        print("  No major positive risk factors (protective profile)")
    print()

print("💡 INSIGHT: Different patients have different risk drivers.")
print("   Personalized explanations enable tailored interventions.\n")

## Part 4: LIME vs SHAP Comparison

Let's compare LIME and SHAP on the same patient.

In [ ]:
import shap

print("Computing SHAP values for comparison...\n")

# Train tree model for TreeSHAP
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_prob = rf_model.predict_proba(X_test)[:, 1]

print(f"Random Forest AUC: {roc_auc_score(y_test, rf_prob):.3f}")

# Compute SHAP
shap_explainer = shap.TreeExplainer(rf_model)
shap_values = shap_explainer.shap_values(X_test)

if isinstance(shap_values, list):
    shap_values_readmit = shap_values[1]
else:
    shap_values_readmit = shap_values

print("✓ SHAP values computed\n")

In [ ]:
# Compare LIME vs SHAP for same patient
compare_idx = high_risk_idx

# LIME explanation (already computed)
lime_exp = explainer.explain_instance(
    X_test[compare_idx],
    rf_model.predict_proba,  # Use same model
    num_features=10,
    num_samples=5000
)

lime_contributions = dict(lime_exp.as_list(label=1))

# SHAP values
shap_contributions = {feat: shap_val for feat, shap_val in 
                      zip(feature_cols, shap_values_readmit[compare_idx])}

# Compare top features
print("="*80)
print(f"LIME vs SHAP: Patient {compare_idx}")
print("="*80 + "\n")

print("Top 5 Features by LIME:\n")
lime_top = sorted(lime_contributions.items(), key=lambda x: abs(x[1]), reverse=True)[:5]
for i, (feat_desc, weight) in enumerate(lime_top, 1):
    # Extract feature name (LIME returns descriptive strings)
    feat_name = feat_desc.split('<=')[0].split('>')[0].strip()
    print(f"  {i}. {feat_name}: {weight:+.3f}")

print("\nTop 5 Features by SHAP:\n")
shap_top = sorted(shap_contributions.items(), key=lambda x: abs(x[1]), reverse=True)[:5]
for i, (feat, shap_val) in enumerate(shap_top, 1):
    print(f"  {i}. {feat}: {shap_val:+.3f}")

print("\n💡 COMPARISON INSIGHTS:")
print("\n  ✓ Both methods typically agree on top features")
print("  ✓ LIME: Linear approximation (intuitive coefficients)")
print("  ✓ SHAP: Exact Shapley values (theoretically grounded)")
print("  ✓ SHAP values sum to prediction difference (additive)")
print("  ✓ LIME faster for large datasets; SHAP more consistent\n")

## Part 5: Clinical Decision Support Interface

Design explanation display for clinicians.

In [ ]:
def generate_clinical_report(patient_idx, patient_features, patient_prob, explanation, feature_cols):
    """
    Generate clinician-friendly readmission risk report.
    """
    print("\n" + "="*80)
    print("READMISSION RISK REPORT")
    print("="*80 + "\n")
    
    print(f"Patient ID: {patient_idx}")
    print(f"30-Day Readmission Risk: {patient_prob:.1%}")
    
    if patient_prob > 0.50:
        risk_cat = "HIGH RISK"
        action = "Recommend intensive discharge planning and early follow-up"
    elif patient_prob > 0.30:
        risk_cat = "MODERATE RISK"
        action = "Ensure outpatient follow-up within 7 days"
    else:
        risk_cat = "LOW RISK"
        action = "Standard discharge planning"
    
    print(f"Risk Category: {risk_cat}")
    print(f"Recommended Action: {action}\n")
    
    print("KEY RISK FACTORS:\n")
    
    lime_values = explanation.as_list(label=1)
    top_risks = [(f, w) for f, w in lime_values if w > 0]
    top_risks_sorted = sorted(top_risks, key=lambda x: x[1], reverse=True)[:5]
    
    for i, (feat_desc, weight) in enumerate(top_risks_sorted, 1):
        print(f"{i}. {feat_desc}")
        
        # Actionable recommendations
        if 'a1c' in feat_desc.lower():
            print("   → Action: Intensify diabetes management, consider endocrine consult")
        elif 'medication' in feat_desc.lower() and 'high' in feat_desc.lower():
            print("   → Action: Medication reconciliation, simplify regimen if possible")
        elif 'admission' in feat_desc.lower():
            print("   → Action: Address underlying causes, ensure continuity of care")
        elif 'insurance' in feat_desc.lower():
            print("   → Action: Social work consult, connect with community resources")
        elif 'dka' in feat_desc.lower():
            print("   → Action: Diabetes education, close monitoring, ensure insulin access")
        print()
    
    # Patient profile
    print("PATIENT PROFILE:\n")
    key_features = ['age', 'a1c', 'num_medications', 'num_prior_admissions', 'has_insurance']
    for feat in key_features:
        if feat in feature_cols:
            idx = feature_cols.index(feat)
            val = patient_features[idx]
            
            if feat == 'has_insurance':
                display = 'Yes' if val == 1 else 'No ⚠️'
            elif feat in ['num_medications', 'num_prior_admissions']:
                display = f"{int(val)}"
            else:
                display = f"{val:.1f}"
            
            print(f"  {feat.replace('_', ' ').title()}: {display}")
    
    print("\n" + "="*80)
    print("\n✓ This report helps clinicians:")
    print("  • Understand WHY the patient is high risk")
    print("  • Identify ACTIONABLE interventions")
    print("  • Prioritize patients for intensive discharge planning")
    print("  • Communicate risk to patients and families\n")

# Generate report
generate_clinical_report(
    patient_idx=patient_idx,
    patient_features=patient_features,
    patient_prob=patient_prob,
    explanation=explanation,
    feature_cols=feature_cols
)

## Summary and Key Takeaways

### What We Learned

1. **LIME Approach**:
   - Approximates black box locally with simple linear model
   - Fast and intuitive
   - Works with any model (model-agnostic)

2. **LIME vs SHAP**:
   - LIME: Faster, more intuitive, better for demos
   - SHAP: More rigorous, consistent, theoretically grounded
   - Both usually agree on top features
   - Use LIME for quick prototyping; SHAP for production

3. **Clinical Applications**:
   - **Readmission prediction**: Identify high-risk patients for intervention
   - **Actionable insights**: Focus on modifiable risk factors
   - **Personalization**: Different patients have different risk drivers
   - **Communication**: Explain risk to patients and families

4. **Practical Workflow**:
   ```
   1. Train black box model
   2. Initialize LIME explainer on training data
   3. For each patient:
      - Generate LIME explanation
      - Extract top contributors
      - Format as clinical report
   4. Validate: Do explanations make clinical sense?
   5. Deploy with real-time explanation interface
   ```

### Best Practices

✅ **Check fidelity**: Ensure LIME's local model accurately approximates black box  
✅ **Use enough samples**: 5000+ perturbed samples for stable explanations  
✅ **Limit features**: Show top 5-10 to avoid overwhelming clinicians  
✅ **Make actionable**: Link features to clinical interventions  
✅ **Iterate with users**: Test explanations with actual clinicians  

### Limitations

⚠️ **Instability**: LIME can give different explanations on repeated calls (randomness in sampling)  
⚠️ **Approximation error**: Linear model may not capture complex local patterns  
⚠️ **No guarantees**: Unlike SHAP, LIME doesn't have theoretical consistency properties  
⚠️ **Parameter sensitive**: Results depend on kernel width, number of samples  

### When to Use LIME

✅ Quick prototyping and demos  
✅ Explaining to non-technical stakeholders  
✅ Black box APIs (no model access)  
✅ Image/text data (LIME excels here)  
✅ Need fast explanations  

❌ Production systems requiring consistency  
❌ Regulatory submissions (prefer SHAP's guarantees)  
❌ Research requiring reproducibility  

---

**Next**: Notebook 10.3 will cover GradCAM for medical imaging—explaining CNNs with visual heatmaps.

*"The best explanation is one that a clinician can act on."*